# DressMe · Objectif 1 — service de reconnaissance (version d'intégration)
`crop partagé → agent (Qwen2.5-VL + FashionCLIP/kNN/RAG) → ItemCard v1 + embedding .npy`

Prérequis : GPU T4, dossier `dressme_obj1/` (ce dépôt) copié dans Drive ou décompressé dans `/content`. Chaque cellule est indépendante de la précédente sauf la cellule 2 (chargement).

In [ ]:
# 0. Installation
!pip -q install -U transformers accelerate bitsandbytes kagglehub tqdm matplotlib pydantic scikit-learn

In [ ]:
# 1. Chemins et configuration (tout se règle ici)
import os, sys
from pathlib import Path
REPO = Path('/content/drive/MyDrive/dressme_obj1')          # <- adapte si le dépôt est ailleurs
from google.colab import drive; drive.mount('/content/drive')
os.environ['DRESSME_ROOT'] = '/content/drive/MyDrive/DressMe_Objective1'   # même dossier de travail que le notebook v1
# os.environ['DRESSME_TYPE_DICT'] = '/content/drive/MyDrive/.../obj6_type_dictionary.json'   # dictionnaire warmth/formality d'Amir (dès qu'il existe)
# os.environ['DRESSME_SHARED_EMBED'] = 'common.embed:embed_image'                               # script FashionCLIP commun de Suzana (dès qu'il existe)
sys.path.insert(0, str(REPO))
from recognition.config import CFG
from contracts import load_type_dictionary
CFG.root = Path(os.environ['DRESSME_ROOT'])
print('dictionnaire officiel chargé :', load_type_dictionary())

In [ ]:
# 2. Chargement (Qwen 4 bits + FashionCLIP gelé + base kNN). La base est reconstruite UNE fois (ajout de article_type).
from recognition.service import RecognitionService
svc = RecognitionService.from_config(CFG)
print('prêt | base kNN :', len(svc.kb_df), '| test catalogue :', len(svc.test_df))

In [ ]:
# 3. Une image DÉJÀ DÉCOUPÉE (crop d'Amir) -> fiche ItemCard
import json
STORE = CFG.root / 'store'           # contient items/ et emb/ (partagé avec l'équipe)
r = svc.recognize('/content/veste_crop.png', item_id='it_0001', out_dir=STORE)
print(r.status, '|', r.category, '| raisons :', r.reasons)
print(r.card.model_dump_json(indent=2) if r.card else 'aucune fiche')
print('\n'.join(r.trace))

In [ ]:
# 4. Évaluation DEV sur les 120 images de catalogue + ablations (appels LLM en cache : presque gratuit)
from eval.eval_catalogue import run_catalogue_eval
run_catalogue_eval(svc, 'v1')
run_catalogue_eval(svc, 'llm_only', 'llm_only')
run_catalogue_eval(svc, 'knn_only', 'knn_only')
svc.cfg.after_rag_policy = 'llm'; run_catalogue_eval(svc, 'after_rag_llm'); svc.cfg.after_rag_policy = 'knn'
# Règle : on ne change CONF_MIN / KNN_MIN / H_HIGH / after_rag_policy QUE d'après ce jeu DEV, jamais d'après les 30 photos réelles.

In [ ]:
# 5. LIVRABLE : 30 photos réelles (crops) étiquetées -> précision slot / catégorie / couleur, F1 macro, IC 95 %, 3 systèmes
from eval.eval_real import run_real_eval
out = run_real_eval(svc, CFG.root / 'real30' / 'labels.csv')   # format : eval/labels_template.csv

In [ ]:
# 6. Export pour l'équipe : un dossier de crops -> fiches JSON + .npy (+ liste de ce qui doit être confirmé par l'utilisateur)
import glob
for i, p in enumerate(sorted(glob.glob(str(CFG.root / 'real30' / '*_crop.png')))):
    res = svc.recognize(p, item_id=f'it_{i+1:04d}', out_dir=STORE)
    print(f'it_{i+1:04d}', res.status, res.category, res.reasons)

In [ ]:
# 7. Mock (aucun modèle) + schéma JSON gelé pour le dossier contracts/ du dépôt commun
from recognition.mock import MockRecognizer
print(MockRecognizer().recognize('x.png').card.model_dump_json(indent=2)[:400])
from contracts import export_json_schema
export_json_schema(REPO / 'contracts' / 'schema' / 'itemcard_v1.schema.json')